# ACIC 2016

This notebook assesses whether TFM-based base learners improve causal machine learning performance within meta-learning frameworks, compared to conventional baseline methods.

**Dataset**: ACIC 2016 (Atlantic Causal Inference Conference), a semi-synthetic benchmark with simulated potential outcomes on real covariate data.
- 10 instances loaded via `causallib`, each with a fixed 80/20 stratified train/test split
- ~4302 train / ~1076 test samples per instance, 58 covariates
- Outcome = simulated continuous response; true ITE available from `mu0`/`mu1`

**Evaluation protocol**:
- Pre-split train/test sets per instance (stratified on treatment)
- All models trained on training set only, evaluated on held-out test set
- R-learner (NonParamDML) and DR-learner (DRLearner) use 5-fold cross-fitting within the training data for nuisance estimation
- LightGBM hyperparameters tuned once per instance via GridSearchCV (27 combinations, 3-fold CV)

**Metrics** (all computed on the test set per instance):
- **PEHE**: Precision in Estimating Heterogeneous Effects — RMSE between predicted and true ITE. Lower is better.
- **ATE Error**: Absolute difference between mean predicted ITE and mean true ITE. Lower is better.

**Models**:
- Meta-learners: S, T, X, R (NonParamDML, cv=5), DR (DRLearner, cv=5)
- Base models: LinearRegression, LightGBM (tuned), TabPFN, TabICL
- Standalone: CausalForestDML (LightGBM nuisance), CausalPFN

## 1. Setup and Imports

In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"  # Ignore warnings
os.environ["TABPFN_NO_TELEMETRY_PROMPT"] = "1"        # Skip TabPFN prompt
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ── IMPORTS ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold, train_test_split
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
import matplotlib.pyplot as plt
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
from sklearn.metrics import mean_squared_error
import warnings
import torch
import tabpfn
from causalpfn import CATEEstimator
from collections import defaultdict
import time


# We use TabPFN 2.6, which corresponds to GitHub 7.x versions
print(tabpfn.__version__)


# ── GLOBAL SEED ──────────────────────────────────────────────────────────────
SEED = 42


# ── DEVICE DETECTION ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

# CausalPFN device: CUDA or CPU
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL device
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")


# ── EXTRA ─────────────────────────────────────────────────────────
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # no-op if no CUDA
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message=".*feature names.*")

### Importing tabpfn v 2.5 alongside v 2.6

In [ ]:
# ── Import TabPFN 2.5 alongside 2.6 ────────────────────────────────────────
import subprocess, sys as _sys

TABPFN25_DIR = "./tabpfn_v25_install"
os.makedirs(TABPFN25_DIR, exist_ok=True)

# Install tabpfn 2.5 to isolated directory if not already present
_tabpfn25_installed = any("tabpfn" in d for d in os.listdir(TABPFN25_DIR))
if not _tabpfn25_installed:
    print("Installing TabPFN 2.5 to isolated directory...")
    subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "tabpfn==6.4.1",
        f"--target={TABPFN25_DIR}", "--quiet", "--no-deps"
    ])
    print("Done.")

# Save current tabpfn 2.6 module references
_tabpfn26_mods = {k: v for k, v in _sys.modules.items()
                  if k == "tabpfn" or k.startswith("tabpfn.")}

# Temporarily inject 2.5 path and load its classes
_sys.path.insert(0, TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]

import tabpfn as _tabpfn25
TabPFNRegressor25 = _tabpfn25.TabPFNRegressor
TabPFNClassifier25 = _tabpfn25.TabPFNClassifier
TABPFN25_VERSION = _tabpfn25.__version__

# Restore tabpfn 2.6
_sys.path.remove(TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]
_sys.modules.update(_tabpfn26_mods)

print(f"TabPFN 2.5 version: {TABPFN25_VERSION}")
print(f"TabPFN 2.6 version: {tabpfn.__version__}")

## 2. Define Models, Tuning, and Metrics

We define LightGBM hyperparameter tuning functions (using `GridSearchCV`) and evaluation metrics. LightGBM is tuned **once per instance** — the best hyperparameters are then reused across all meta-learners for that instance.

In [ ]:
# ── TUNING ─────────────────────────────────────────────────────────
# LightGBM hyperparameter search space (3×3×3 = 27 combinations — full grid)
LGBM_GRID = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [20, 50, 100],
    'n_estimators':      [200, 500, 1000],
}

NUISANCE_CV = 5     # K-fold cross-fitting for R- and DR-learner nuisance models

# LightGBM Tuning
def tune_lgbm(X, y, classifier=False, stratify=None, seed=SEED):
    """Tune LGBM via GridSearchCV. Returns best_params_ dict.

    - classifier = False  → LGBMRegressor, scored by neg_MSE
    - classifier = True   → LGBMClassifier, scored by neg_log_loss
    - stratify          → use StratifiedKFold on this variable (regression only);
                          if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)

    if classifier:
        # Propensity Model (classification)
        base    = LGBMClassifier(random_state=seed, verbose=-1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    else:
        # Outcome Model (regression): S-, R-, and DR-learner
        base    = LGBMRegressor(random_state=seed, verbose=-1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, stratify))
        else:
            # Single Treatment Arm: T- and X-learner
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle=True, random_state=seed).split(X))

    search = GridSearchCV(
        base, LGBM_GRID, scoring=scoring,
        cv=cv, n_jobs=-1,
    )

    search.fit(X, y)
    return search.best_params_

# Pseudo-Outcome Tuning: X-, R-, and DR-learner final-stage models
def make_lgbm_final(seed=SEED):
    """LGBM wrapped in GridSearchCV for final-stage tuning on pseudo-outcomes."""
    return GridSearchCV(
        LGBMRegressor(random_state=seed, verbose=-1),
        LGBM_GRID, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1,
    )


# ── Evaluation Metric functions ───────────────────────────────────────────────
def calculate_pehe(predicted_ite, true_ite):
    return np.sqrt(mean_squared_error(true_ite, predicted_ite))

def calculate_ate_error(predicted_ite, true_ite):
    return np.abs(predicted_ite.mean() - true_ite.mean())

## 3. Data Loading (ACIC 2016)

We load all 10 instances of the ACIC 2016 dataset via causallib and store them for evaluation. Each instance is split 80/20 into train/test (stratified on treatment).

In [ ]:
# ── Data Loading ─────────────────────────────────────────────────────────
# Store results for all runs
# Structure: results[meta_learner][base_model][metric] = list of values
all_results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

from causallib.datasets import load_acic16

n_datasets = 10  # Use all 10 ACIC 2016 instances
processed_datasets = []

print(f"Loading {n_datasets} ACIC 2016 instances via causallib...")

for i in range(1, n_datasets + 1):
    data = load_acic16(instance=i)
    X_full   = data.X.reset_index(drop=True)
    T_full   = data.a.values
    Y_full   = data.y.values
    mu0_full = data.po['0'].values
    mu1_full = data.po['1'].values

    # Stratified 80/20 train/test split (stratified on treatment)
    idx_train, idx_test = train_test_split(
        np.arange(len(T_full)),
        test_size=0.2,
        random_state=i,
        stratify=T_full
    )

    X_train = X_full.iloc[idx_train].reset_index(drop=True)
    X_test  = X_full.iloc[idx_test].reset_index(drop=True)
    T_train = T_full[idx_train]
    T_test  = T_full[idx_test]
    Y_train = Y_full[idx_train]
    Y_test  = Y_full[idx_test]
    true_ITE_test = mu1_full[idx_test] - mu0_full[idx_test]

    processed_datasets.append({
        'id':            i,
        'X_train':       X_train,
        'X_test':        X_test,
        'T_train':       T_train,
        'T_test':        T_test,
        'Y_train':       Y_train,
        'Y_test':        Y_test,
        'true_ITE_test': true_ITE_test,
    })


# ── Data Summary ─────────────────────────────────────────────────────────
rep = processed_datasets[0]
X_tr, T_tr, Y_tr = rep['X_train'], rep['T_train'], rep['Y_train']
X_te, T_te, Y_te = rep['X_test'],  rep['T_test'],  rep['Y_test']

n_train, n_test = len(T_tr), len(T_te)
n_total   = n_train + n_test
n_treated = int(T_tr.sum()) + int(T_te.sum())
n_control = n_total - n_treated
ate_naive = Y_tr[T_tr == 1].mean() - Y_tr[T_tr == 0].mean()
true_ate  = rep['true_ITE_test'].mean()

Y_all_rep = np.concatenate([Y_tr, Y_te])

print("=" * 55)
print("ACIC 2016 Dataset Summary (instance 1)")
print("=" * 55)
print(f"  Instances      : {n_datasets}")
print(f"  Train samples  : {n_train}  |  Test samples: {n_test}")
print(f"  Total (inst 1) : {n_total}")
print(f"  Treated        : {n_treated}  ({100*n_treated/n_total:.1f}%)")
print(f"  Control        : {n_control}  ({100*n_control/n_total:.1f}%)")
print(f"  Features       : {X_tr.shape[1]}")
print(f"  Outcome (Y)    : range [{Y_all_rep.min():.2f}, {Y_all_rep.max():.2f}]")
print(f"                   mean  {Y_all_rep.mean():.2f}  (std {Y_all_rep.std():.2f})")
print(f"  ATE naive      : {ate_naive:+.4f}  (treated mean - control mean, train)")
print(f"  True ATE       : {true_ate:+.4f}  (mean true ITE on test set)")
print()
print("Outcome by treatment arm (train, inst 1):")
print(f"  Treated  mean Y: {Y_tr[T_tr==1].mean():.4f}  (std {Y_tr[T_tr==1].std():.4f})")
print(f"  Control  mean Y: {Y_tr[T_tr==0].mean():.4f}  (std {Y_tr[T_tr==0].std():.4f})")

print(f"\nLoaded {len(processed_datasets)} instances.")
print(f"Train size: {X_tr.shape[0]}, Test size: {X_te.shape[0]}, Features: {X_tr.shape[1]}")

## 4. Unified Meta-Learner Evaluation

We evaluate all five meta-learners (S, T, X, R, DR) in a single loop over the 10 ACIC instances. For each instance, we tune LightGBM once for the outcome model (regressor) and once for the propensity model (classifier), then reuse those tuned models across all meta-learners. This avoids redundant tuning and ensures consistency.

In [ ]:
# ── Evaluation Prep ─────────────────────────────────────────────────────────
print("Evaluating all meta-learners across 10 ACIC instances...\n")

for dataset in processed_datasets:
    i = dataset['id']
    n = len(processed_datasets)
    print(f"\n{'='*70}")
    print(f" Dataset {i}/{n}")
    print(f"{'='*70}")

    X_train, X_test = dataset['X_train'], dataset['X_test']
    T_train         = dataset['T_train']
    Y_train         = dataset['Y_train']
    true_ITE_test   = dataset['true_ITE_test']


    # ── Arm splits (needed for T- and X-learner tuning) ──────────────────────
    ctrl = T_train == 0
    trt  = T_train == 1
    X_ctrl, Y_ctrl = X_train[ctrl], Y_train[ctrl]
    X_trt,  Y_trt  = X_train[trt],  Y_train[trt]


    # ── LightGBM hyperparameter tuning ───────────────────────────────────────
    t0 = time.time()
    X_with_T       = np.column_stack([X_train, T_train])
    params_s       = tune_lgbm(X_with_T, Y_train, stratify=T_train)  # S-learner outcome (X+T features)
    params_outcome = tune_lgbm(X_train,  Y_train, stratify=T_train)  # outcome nuisance (R-learner model_y)
    params_prop    = tune_lgbm(X_train,  T_train, classifier=True)   # propensity model
    params_ctrl    = tune_lgbm(X_ctrl,   Y_ctrl)                     # T/X control arm outcome
    params_trt     = tune_lgbm(X_trt,    Y_trt)                      # T/X treated arm outcome
    print(f"  LightGBM tuning: {time.time() - t0:.1f}s")


    # ── Model configs ────────────────────────────────────────────────────────
    # Local shorthands to avoid repeating constructor args across 4 models x 11 roles
    def _lgbm_r(params):    return LGBMRegressor(random_state=SEED, verbose=-1, **params)
    def _lgbm_c(params):    return LGBMClassifier(random_state=SEED, verbose=-1, **params)
    def _tabpfn_r():        return TabPFNRegressor(device=device, random_state=SEED)
    def _tabpfn_c():        return TabPFNClassifier(device=device, random_state=SEED)
    def _tabicl_r():        return TabICLRegressor(device=tabicl_device, random_state=SEED, verbose=False)
    def _tabicl_c():        return TabICLClassifier(device=tabicl_device, random_state=SEED, verbose=False)
    def _tabpfn25_r():      return TabPFNRegressor25(device=device, random_state=SEED)
    def _tabpfn25_c():      return TabPFNClassifier25(device=device, random_state=SEED)

    base_model_configs = {
        'LinearRegression': {
            's_model':       LinearRegression(),
            't_models':      (LinearRegression(), LinearRegression()),
            'x_models':      (LinearRegression(), LinearRegression()),
            'x_cate':        (LinearRegression(), LinearRegression()),
            'x_propensity':  LogisticRegression(max_iter=1000, random_state=SEED),
            'r_model_y':     LinearRegression(),
            'r_model_t':     LogisticRegression(max_iter=1000, random_state=SEED),
            'r_model_final': LinearRegression(),
            'dr_regression': LinearRegression(),
            'dr_propensity': LogisticRegression(max_iter=1000, random_state=SEED),
            'dr_final':      LinearRegression(),
        },
        'LightGBM': {
            's_model':       _lgbm_r(params_s),
            't_models':      (_lgbm_r(params_ctrl), _lgbm_r(params_trt)),
            'x_models':      (_lgbm_r(params_ctrl), _lgbm_r(params_trt)),
            'x_cate':        (make_lgbm_final(), make_lgbm_final()),
            'x_propensity':  _lgbm_c(params_prop),
            'r_model_y':     _lgbm_r(params_outcome),
            'r_model_t':     _lgbm_c(params_prop),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _lgbm_r(params_s),
            'dr_propensity': _lgbm_c(params_prop),
            'dr_final':      make_lgbm_final(),
        },
        'TabPFN_v2.5': {
            's_model':       _tabpfn25_r(),
            't_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_cate':        (_tabpfn25_r(), _tabpfn25_r()),
            'x_propensity':  _tabpfn25_c(),
            'r_model_y':     _tabpfn25_r(),
            'r_model_t':     _tabpfn25_c(),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _tabpfn25_r(),
            'dr_propensity': _tabpfn25_c(),
            'dr_final':      _tabpfn25_r(),
        },
        'TabPFN_v2.6': {
            's_model':       _tabpfn_r(),
            't_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_cate':        (_tabpfn_r(), _tabpfn_r()),
            'x_propensity':  _tabpfn_c(),
            'r_model_y':     _tabpfn_r(),
            'r_model_t':     _tabpfn_c(),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _tabpfn_r(),
            'dr_propensity': _tabpfn_c(),
            'dr_final':      _tabpfn_r(),
        },
        'TabICL': {
            's_model':       _tabicl_r(),
            't_models':      (_tabicl_r(), _tabicl_r()),
            'x_models':      (_tabicl_r(), _tabicl_r()),
            'x_cate':        (_tabicl_r(), _tabicl_r()),
            'x_propensity':  _tabicl_c(),
            'r_model_y':     _tabicl_r(),
            'r_model_t':     _tabicl_c(),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _tabicl_r(),
            'dr_propensity': _tabicl_c(),
            'dr_final':      _tabicl_r(),
        },
    }

    # ── Evaluation ─────────────────────────────────────────────────────────
    for name, cfg in base_model_configs.items():

        # ── S-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        s_learner = SLearner(overall_model=cfg['s_model'])
        s_learner.fit(Y_train, T_train, X=X_train)
        te = s_learner.effect(X_test)

        all_results['S'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['S'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  S-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── T-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        t_learner = TLearner(models=cfg['t_models'])
        t_learner.fit(Y_train, T_train, X=X_train)
        te = t_learner.effect(X_test)

        all_results['T'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['T'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  T-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── X-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        x_learner = XLearner(models=cfg['x_models'], cate_models=cfg['x_cate'], propensity_model=cfg['x_propensity'])
        x_learner.fit(Y_train, T_train, X=X_train)
        te = x_learner.effect(X_test)

        all_results['X'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['X'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  X-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── R-Learner (NonParamDML) ────────────────────────────────────────────
        t0 = time.time()
        r_learner = NonParamDML(
            model_y=cfg['r_model_y'], model_t=cfg['r_model_t'],
            model_final=cfg['r_model_final'], discrete_treatment=True, cv=NUISANCE_CV, random_state=SEED
        )
        r_learner.fit(Y_train, T_train, X=X_train)
        te = r_learner.effect(X_test)

        all_results['R'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['R'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  R-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── DR-Learner ────────────────────────────────────────────────────────
        t0 = time.time()
        dr_learner = DRLearner(
            model_regression=cfg['dr_regression'],
            model_propensity=cfg['dr_propensity'],
            model_final=cfg['dr_final'],
            min_propensity=0.05,  # prevents extreme IPW weights
            cv=NUISANCE_CV,
            random_state=SEED
        )
        dr_learner.fit(Y_train, T_train, X=X_train)
        te = dr_learner.effect(X_test)

        all_results['DR'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['DR'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  DR-learner + {name:20s}: {time.time() - t0:.1f}s")


    # ── Causal Forest (CausalForestDML) ───────────────────────────────────────
    try:
        t0 = time.time()
        cf = CausalForestDML(
            model_y=LGBMRegressor(random_state=SEED, verbose=-1, **params_outcome),
            model_t=LGBMClassifier(random_state=SEED, verbose=-1, **params_prop),
            discrete_treatment=True,
            cv=NUISANCE_CV,
            n_estimators=500,
            min_samples_leaf=5,
            random_state=SEED,
        )
        cf.tune(Y_train, T_train, X=X_train)
        cf.fit(Y_train, T_train, X=X_train)
        te = cf.effect(X_test)

        all_results['CF']['CausalForest']['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['CF']['CausalForest']['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  CF         + {'CausalForest':20s}: {time.time() - t0:.1f}s")
    except Exception as e:
        print(f"\nCausalForest error on dataset {i}: {e}")


    # ── CausalPFN ─────────────────────────────────────────────────────────────
    try:
        print(f"  [CausalPFN]...", end=" ", flush=True)
        _t0 = time.time()
        cpfn = CATEEstimator(device=causalpfn_device, verbose=False)

        X_cpfn_train = np.asarray(X_train, dtype=np.float32)
        T_cpfn_train = np.asarray(T_train, dtype=np.float32).ravel()
        Y_cpfn_train = np.asarray(Y_train, dtype=np.float32).ravel()
        X_cpfn_test  = np.asarray(X_test,  dtype=np.float32)

        cpfn.fit(X_cpfn_train, T_cpfn_train, Y_cpfn_train)
        te = cpfn.estimate_cate(X_cpfn_test)

        if "torch" in str(type(te)):
            te = te.detach().cpu().numpy()

        te = np.asarray(te, dtype=np.float32).reshape(-1)

        all_results["CausalPFN"]["CausalPFN"]["pehe"].append(calculate_pehe(te, true_ITE_test))
        all_results["CausalPFN"]["CausalPFN"]["ate_error"].append(calculate_ate_error(te, true_ITE_test))
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalPFN error on dataset {i}: {exc}")

print("\nAll meta-learner evaluations complete.")

## 5. Aggregated Results

We report the Mean and Standard Error of PEHE and ATE Error across the 10 instances.

In [ ]:
# ── Results ─────────────────────────────────────────────────────────
summary_rows = []

for meta in ['S', 'T', 'X', 'R', 'DR', 'CF', 'CausalPFN']:
    if meta == 'CausalPFN':
        models = ['CausalPFN']
    elif meta == 'CF':
        models = ['CausalForest']
    else:
        base_models = ['LinearRegression', 'LightGBM', 'TabPFN_v2.6', 'TabPFN_v2.5', 'TabICL']
        models = base_models

    for model in models:
        pehes = all_results[meta][model]['pehe']
        ate_errs = all_results[meta][model]['ate_error']

        if not pehes:
            continue

        n = len(pehes)
        summary_rows.append({
            'Meta-Learner': meta,
            'Base Model': model,
            'PEHE Mean': np.mean(pehes),
            'PEHE SE': np.std(pehes, ddof=1) / np.sqrt(n),
            'ATE Error Mean': np.mean(ate_errs),
            'ATE Error SE': np.std(ate_errs, ddof=1) / np.sqrt(n)
        })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# Export to CSV
csv_path = 'benchmark_results_ACIC.csv'
df_summary.to_csv(csv_path, index=False)
print(f"\nResults exported to {csv_path}")

# --- Save results to disk ---
import pickle

_save_path = 'benchmark_results_ACIC.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump({
        'all_results': {m: {n: dict(d) for n, d in v.items()} for m, v in all_results.items()},
        'NUISANCE_CV': NUISANCE_CV,
        'n_datasets': n_datasets,
    }, f)
print(f"Results saved to {_save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Colors per meta-learner ────────────────────────────────────────────────
COLOR_MAP = {
    'DR': '#1f77b4',   # blue
    'R':  '#ff7f0e',   # orange
    'S':  '#2ca02c',   # green
    'T':  '#d62728',   # red
    'X':  '#9467bd',   # purple
}
STANDALONE_COLORS = {
    'CF':        '#00008B',  # dark blue
    'CausalPFN': '#8B0000',  # dark red
}
STANDALONE_LABELS = {
    'CF':        'CausalForest',
    'CausalPFN': 'CausalPFN',
}

meta_learners = ['DR', 'R', 'S', 'T', 'X']
base_models   = ['LightGBM', 'LinearRegression', 'TabICL', 'TabPFN_v2.5', 'TabPFN_v2.6']
base_labels   = ['LightGBM', 'LinearRegression', 'TabICL', 'TabPFN v2.5', 'TabPFN v2.6']
metrics       = ['PEHE', 'ATE Error']
mean_cols     = ['PEHE Mean', 'ATE Error Mean']
se_cols       = ['PEHE SE',   'ATE Error SE']

n_groups  = len(base_models)
n_bars    = len(meta_learners)
bar_width = 0.14
x         = np.arange(n_groups)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (metric, mean_col, se_col) in enumerate(zip(metrics, mean_cols, se_cols)):
    ax = axes[idx]

    # ── Grouped bars ──────────────────────────────────────────────────────
    for j, meta in enumerate(meta_learners):
        means, ses = [], []
        for bm in base_models:
            row = df_summary[
                (df_summary['Meta-Learner'] == meta) &
                (df_summary['Base Model']   == bm)
            ]
            if row.empty:
                means.append(0); ses.append(0)
            else:
                means.append(row[mean_col].values[0])
                ses.append(row[se_col].values[0])

        offset = (j - (n_bars - 1) / 2) * bar_width
        ax.bar(
            x + offset, means,
            width=bar_width,
            color=COLOR_MAP[meta],
            label=meta,
            zorder=3,
        )
        ax.errorbar(
            x + offset, means,
            yerr=ses,
            fmt='none',
            ecolor='black',
            capsize=3,
            linewidth=1,
            zorder=4,
        )

    # ── Dashed horizontal lines for CausalForest and CausalPFN ───────────
    for meta, color in STANDALONE_COLORS.items():
        row = df_summary[df_summary['Meta-Learner'] == meta]
        if row.empty:
            continue
        val   = row[mean_col].values[0]
        label = f"{STANDALONE_LABELS[meta]} ({val:.3f})"
        ax.axhline(y=val, linestyle='--', linewidth=1.5, color=color, label=label, zorder=1)

    # ── Styling ───────────────────────────────────────────────────────────
    ax.set_title(f'{metric} (Mean ± SE)')
    ax.set_ylabel(metric)
    ax.set_xlabel('Base Model')
    ax.set_xticks(x)
    ax.set_xticklabels(base_labels, rotation=15, ha='right')
    ax.grid(True, alpha=0.3, axis='y', zorder=0)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)

plt.suptitle('Results on ACIC 2016', fontsize=13)
plt.tight_layout()
plt.savefig("results_acic_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()